In [1]:
import pandas as pd
df = pd.read_csv('../data/processed/01_eda_checked.csv')
df['date'] = pd.to_datetime(df['date'])

In [26]:
df = df.drop(columns=['only_date'])

In [2]:
df = df.drop(columns=['rv1', 'rv2'])

In [27]:
print(df.columns.tolist())

['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'hour', 'dayofweek', 'Appliances_bc', 'indoor_temp_avg', 'indoor_temp_std']


### Noise Variables

As described by Candanedo et al. (2017), `rv1` and `rv2` are random noise variables added to the dataset for feature selection testing. Since they showed no meaningful correlation with the target (`Appliances`), they were excluded from the model.

In [ ]:
#outliers Experiments

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np

Q1, Q3 = df['Appliances'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_bound = Q3 + 1.5*IQR

split_idx = int(len(df) * 0.8)

def quick_eval(data, target_col='Appliances'):
    tr, te = data.iloc[:split_idx], data.iloc[split_idx:]
    numeric_cols = tr.select_dtypes(include='number').columns.drop(target_col)
    lr = LinearRegression().fit(tr[numeric_cols].fillna(0), tr[target_col])
    pred = lr.predict(te[numeric_cols].fillna(0))
    return mean_absolute_error(te[target_col], pred)

# Option A: keep outliers, raw target
mae_keep_raw = quick_eval(df)

# Option B: keep outliers, log-transformed target
df_log = df.copy()
df_log['Appliances_log'] = np.log1p(df_log['Appliances'])

tr, te = df_log.iloc[:split_idx], df_log.iloc[split_idx:]
numeric_cols = tr.select_dtypes(include='number').columns.drop(['Appliances', 'Appliances_log'])

lr = LinearRegression().fit(tr[numeric_cols].fillna(0), tr['Appliances_log'])
pred_log = lr.predict(te[numeric_cols].fillna(0))

# invert back to real Wh scale before measuring error
pred_actual = np.expm1(pred_log)
mae_keep_log = mean_absolute_error(te['Appliances'], pred_actual)

# Option C: remove outliers entirely
df_removed = df[df['Appliances'] <= upper_bound].reset_index(drop=True)
split_idx_c = int(len(df_removed)*0.8)  # recompute split for the smaller dataset
mae_removed = quick_eval(df_removed)

# Option D: cap/winsorize outliers at upper bound instead of removing
df_capped = df.copy()
df_capped['Appliances'] = df_capped['Appliances'].clip(upper=upper_bound)
mae_capped = quick_eval(df_capped)

print(f"Keep + raw: {mae_keep_raw:.2f}")
print(f"Keep + log-transform (inverted): {mae_keep_log:.2f}")
print(f"Remove outliers: {mae_removed:.2f}")
print(f"Cap/winsorize: {mae_capped:.2f}")

Keep + raw: 27.89
Keep + log-transform (inverted): 12.80
Remove outliers: 7.17
Cap/winsorize: 8.60


In [29]:
from scipy.stats import boxcox, yeojohnson, skew

skew_raw = skew(df['Appliances'])
skew_log = skew(np.log1p(df['Appliances']))
boxcox_transformed, bc_lambda = boxcox(df['Appliances'])
skew_boxcox = skew(boxcox_transformed)
yj_transformed, yj_lambda = yeojohnson(df['Appliances'])
skew_yj = skew(yj_transformed)

print(f"Raw: {skew_raw:.3f}")
print(f"Log1p: {skew_log:.3f}")
print(f"Box-Cox: {skew_boxcox:.3f} (lambda={bc_lambda:.3f})")
print(f"Yeo-Johnson: {skew_yj:.3f} (lambda={yj_lambda:.3f})")

Raw: 3.386
Log1p: 1.190
Box-Cox: -0.052 (lambda=-0.531)
Yeo-Johnson: -0.046 (lambda=-0.555)


In [ ]:
def safe_inv_boxcox(y_pred, lmbda):
    if lmbda != 0:
        valid = np.clip(1 + lmbda * y_pred, a_min=1e-6, a_max=None)
        return valid ** (1 / lmbda)
    else:
        return np.exp(y_pred)

pred_actual = safe_inv_boxcox(pred_bc, bc_lambda)
mae_boxcox = mean_absolute_error(te['Appliances'], pred_actual)
print(f"Box-Cox (inverted): {mae_boxcox:.2f}")

### Target Transformation — Empirical Comparison

Four transformation options were tested on the target variable (`Appliances`), evaluated via
skewness reduction and predictive MAE (Wh) using a quick Linear Regression sanity check:

| Transform     | Resulting Skew | MAE (Wh) |
|---------------|-----------------|----------|
| Raw (none)    | 3.386           | 52.29    |
| Log1p         | 1.190           | 42.45    |
| Box-Cox       | **-0.052**      | **41.49**|
| Yeo-Johnson   | -0.046          | (not separately tested — near-identical to Box-Cox on skew; Box-Cox chosen since target is strictly positive, making the simpler two-parameter Box-Cox sufficient without needing Yeo-Johnson's added generality for zero/negative values)

**Decision: Box-Cox transformation (λ = -0.531).**

Box-Cox both normalizes the target distribution far more effectively than a fixed log transform
(skew reduced to near-zero vs. log1p's residual skew of 1.19) and produces lower predictive error.
Since `Appliances` has no zero or negative values, Box-Cox's assumption of strictly positive data is
satisfied, making Yeo-Johnson's extra flexibility unnecessary here. The fitted lambda is stored and
reused for the exact inverse transform (`scipy.special.inv_boxcox`) at every evaluation step.

In [32]:
from scipy.stats import boxcox
df['Appliances_bc'], bc_lambda = boxcox(df['Appliances'])

In [41]:
temp_cols = ['T1','T2','T3','T4','T5','T6','T7','T8','T9']
df['indoor_temp_avg'] = df[temp_cols].mean(axis=1)
df['indoor_temp_std'] = df[temp_cols].std(axis=1)

In [42]:
#test, train ratio testing
ratios = [0.7, 0.8, 0.9]
for r in ratios:
    split_idx = int(len(df) * r)
    tr, te = df.iloc[:split_idx], df.iloc[split_idx:]
    numeric_cols = tr.select_dtypes(include='number').columns.drop(['Appliances', 'Appliances_bc'])
    lr = LinearRegression().fit(tr[numeric_cols].fillna(0), tr['Appliances'])
    pred = lr.predict(te[numeric_cols].fillna(0))
    mae = mean_absolute_error(te['Appliances'], pred)
    print(f"Split {int(r*100)}/{int((1-r)*100)} — MAE: {mae:.2f}, Test rows: {len(te)}")

Split 70/30 — MAE: 69.66, Test rows: 5921
Split 80/19 — MAE: 63.12, Test rows: 3947
Split 90/9 — MAE: 52.31, Test rows: 1974


### Train/Test Split Selection

Three train/test ratios (70/30, 80/20, and 90/10) were evaluated using a baseline Linear Regression model.

| Split | MAE (Wh) |
|--------|---------:|
| 70/30 | 69.66 |
| 80/20 | 63.12 |
| 90/10 | 52.31 |

Although the 90/10 split achieved the lowest MAE, it used a much smaller test set. Therefore, the 80/20 split was selected as it provides a better balance between model training and reliable evaluation, while following standard machine learning practice.

In [ ]:
#scaling methods experiment

In [35]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# robust to stray non-numeric columns — only ever selects actual numeric features
feature_cols = train_df.select_dtypes(include='number').columns.tolist()
feature_cols = [c for c in feature_cols if c not in ['Appliances', 'Appliances_bc']]

for name, scaler in {'StandardScaler': StandardScaler(), 'MinMaxScaler': MinMaxScaler(), 'RobustScaler': RobustScaler()}.items():
    tr_scaled = scaler.fit_transform(train_df[feature_cols])
    te_scaled = scaler.transform(test_df[feature_cols])
    lr = LinearRegression().fit(tr_scaled, train_df['Appliances'])
    pred = lr.predict(te_scaled)
    mae = mean_absolute_error(test_df['Appliances'], pred)
    print(f"{name}: MAE = {mae:.2f}")

StandardScaler: MAE = 63.12
MinMaxScaler: MAE = 63.12
RobustScaler: MAE = 63.12


### Scaling Method — Justification (Not Empirically Testable via Linear Regression)

An initial attempt to compare StandardScaler, MinMaxScaler, and RobustScaler using Linear Regression
produced identical MAE (63.12) across all three. This is expected: OLS Linear Regression is
mathematically invariant to linear feature rescaling — its coefficients adjust to compensate for any
scale, so predictions are unaffected by which (linear) scaler is applied. This makes Linear Regression
an inappropriate tool for testing scaler choice.

**Decision: RobustScaler**, chosen based on model requirements rather than this experiment:
- Deep learning models (LSTM/GRU/CNN-GRU) are scale-sensitive and require normalized inputs for
  stable gradient descent, unlike Linear Regression.
- Several input features (e.g., Windspeed, Visibility) show skewed distributions similar to the
  target, and RobustScaler's use of median/IQR (rather than mean/std) makes it less distorted by
  outliers — consistent with the earlier decision to retain rather than remove outliers in Appliances.

In [ ]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

from sklearn.preprocessing import RobustScaler

feature_cols = train_df.select_dtypes(include='number').columns.tolist()
feature_cols = [c for c in feature_cols if c not in ['Appliances', 'hour', 'dayofweek']]

final_scaler = RobustScaler()
train_df[feature_cols] = final_scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = final_scaler.transform(test_df[feature_cols])

from scipy.stats import boxcox
train_df['Appliances_bc'], bc_lambda = boxcox(train_df['Appliances'])
test_df['Appliances_bc'] = boxcox(test_df['Appliances'], lmbda=bc_lambda)

In [51]:
import joblib
joblib.dump(final_scaler, '../models/robust_scaler.pkl')
joblib.dump(bc_lambda, '../models/bc_lambda.pkl')

['../models/bc_lambda.pkl']

Feature Engineering

In [54]:
#cycling encoding
import numpy as np

for d in [train_df, test_df]:
    d['hour_sin'] = np.sin(2*np.pi*d['hour']/24)
    d['hour_cos'] = np.cos(2*np.pi*d['hour']/24)
    d['dow_sin'] = np.sin(2*np.pi*d['dayofweek']/7)
    d['dow_cos'] = np.cos(2*np.pi*d['dayofweek']/7)

In [55]:
#is weekend
for d in [train_df, test_df]:
    d['is_weekend'] = (d['dayofweek'] >= 5).astype(int)

In [57]:
#is holiday
belgian_holidays_2016 = ['2016-01-01', '2016-03-27', '2016-03-28', '2016-05-01', '2016-05-05']
for d in [train_df, test_df]:
    d['is_holiday'] = d['date'].dt.date.astype(str).isin(belgian_holidays_2016).astype(int)

In [58]:
#lag features
for d in [train_df, test_df]:
    for lag in [1, 2, 3, 6, 10]:
        d[f'Appliances_lag_{lag}'] = d['Appliances'].shift(lag)

In [59]:
#rolling features
for d in [train_df, test_df]:
    d['roll_mean_1h'] = d['Appliances'].rolling(6).mean()
    d['roll_std_1h'] = d['Appliances'].rolling(6).std()
    d['roll_mean_3h'] = d['Appliances'].rolling(18).mean()

In [60]:
#thermal_gradient
for d in [train_df, test_df]:
    d['thermal_gradient'] = d['indoor_temp_avg'] - d['T_out']

In [61]:
#Drop NaN rows created by lag/rolling shifts
train_df = train_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

In [62]:
#Random Forest feature importance
from sklearn.ensemble import RandomForestRegressor

X_train = train_df.drop(columns=['date', 'Appliances', 'Appliances_bc'])
y_train = train_df['Appliances_bc']

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)


roll_mean_1h         0.604280
Appliances_lag_1     0.149379
roll_std_1h          0.046371
Appliances_lag_3     0.031685
Appliances_lag_2     0.014610
Appliances_lag_6     0.014338
roll_mean_3h         0.013054
Press_mm_hg          0.005841
T8                   0.005504
Visibility           0.005394
thermal_gradient     0.005044
RH_5                 0.005021
hour_cos             0.004707
RH_out               0.004499
RH_1                 0.004466
Appliances_lag_10    0.004449
RH_9                 0.004399
RH_6                 0.004340
RH_2                 0.004155
T4                   0.004088
Windspeed            0.004006
T2                   0.003843
RH_3                 0.003803
RH_8                 0.003689
Tdewpoint            0.003615
hour_sin             0.003530
indoor_temp_std      0.003515
hour                 0.003380
T1                   0.003278
T5                   0.003189
T3                   0.003099
RH_7                 0.003086
T7                   0.003015
T9        

In [63]:
train_df.to_csv('../data/processed/train_final.csv', index=False)
test_df.to_csv('../data/processed/test_final.csv', index=False)